In [ ]:
from cellpose import models

In [ ]:
import os
import time
import numpy as np
import tifffile
import zarr

from cellpose import models
from tifffile import imwrite

# ============================================================
# CONFIG
# ============================================================

IMAGE_PATH = (
    "../../../Broad_SpatialFoundation/"
    "test_data/10X_Xenium_Ovarian_5k/"
    "morphology_focus/morphology_focus_0000.ome.tif"
)

OUTPUT_DIR = "cellpose_tiles"

TILE_SIZE = 2048
OVERLAP = 256

STEP = TILE_SIZE - 2 * OVERLAP

# Skip tiles with almost no signal
MIN_P99 = 50

# ============================================================
# LOAD IMAGE
# ============================================================

print("Opening image...")

store = tifffile.imread(
    IMAGE_PATH,
    aszarr=True,
)

z = zarr.open(
    store,
    mode="r",
)

print("Image shape:", z.shape)

n_channels, height, width = z.shape

# ============================================================
# MODEL
# ============================================================

print("Loading Cellpose...")

model = models.CellposeModel(
    gpu=True,
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True,
)

# ============================================================
# TILES
# ============================================================

n_tiles_y = int(np.ceil(height / STEP))
n_tiles_x = int(np.ceil(width / STEP))

total_tiles = n_tiles_y * n_tiles_x

print(
    f"{n_tiles_y} x {n_tiles_x} = "
    f"{total_tiles} tiles"
)

tile_counter = 0
processed_tiles = 0

start_time = time.time()

# ============================================================
# LOOP
# ============================================================

for y0 in range(0, height, STEP):

    for x0 in range(0, width, STEP):

        tile_counter += 1

        y1 = min(
            y0 + TILE_SIZE,
            height,
        )

        x1 = min(
            x0 + TILE_SIZE,
            width,
        )

        t0 = time.time()

        # ----------------------------------------------------
        # Read tile
        # ----------------------------------------------------

        patch = z[
            :,
            y0:y1,
            x0:x1,
        ]

        # CYX -> YXC
        patch = np.moveaxis(
            patch,
            0,
            -1,
        )

        # ----------------------------------------------------
        # Quick empty-tile check
        # ----------------------------------------------------

        if np.percentile(
            patch[..., 0],
            99,
        ) < MIN_P99:

            print(
                f"[{tile_counter}/{total_tiles}] "
                f"Skipping empty tile "
                f"({y0},{x0})"
            )

            continue

        # ----------------------------------------------------
        # Build RGB image
        # ----------------------------------------------------

        rgb = np.stack(
            [
                patch[..., 2],
                patch[..., 1],
                patch[..., 0],
            ],
            axis=-1,
        ).astype(np.float32)

        for c in range(3):

            p = np.percentile(
                rgb[..., c],
                99.5,
            )

            if p > 0:
                rgb[..., c] /= p

        rgb = np.clip(
            rgb,
            0,
            1,
        )

        # ----------------------------------------------------
        # Cellpose
        # ----------------------------------------------------

        masks, _, _ = model.eval(
            rgb,
        )

        top = 0 if y0 == 0 else OVERLAP
        left = 0 if x0 == 0 else OVERLAP
        
        bottom = masks.shape[0]
        right = masks.shape[1]
        
        if y1 < height:
            bottom -= OVERLAP
        
        if x1 < width:
            right -= OVERLAP
        
        masks_center = masks[
            top:bottom,
            left:right,
        ]

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        out_path = os.path.join(
            OUTPUT_DIR,
            f"mask_y{y0}_x{x0}_top{top}_left{left}.tif",
        )

        imwrite(
            out_path,
            masks_center.astype(np.uint32),
        )

        processed_tiles += 1

        tile_time = time.time() - t0

        elapsed = time.time() - start_time

        avg_time = elapsed / processed_tiles

        remaining = (
            total_tiles - tile_counter
        )

        eta_min = (
            avg_time * remaining
        ) / 60

        print(
            f"[{tile_counter}/{total_tiles}] "
            f"Done tile "
            f"({y0},{x0}) | "
            f"cells={masks.max()} | "
            f"{tile_time:.1f}s | "
            f"ETA={eta_min:.1f} min"
        )

# ============================================================
# DONE
# ============================================================

total_time = (
    time.time() - start_time
) / 60

print()
print("=" * 60)
print("Finished!")
print(f"Processed tiles: {processed_tiles}")
print(f"Total time: {total_time:.1f} min")
print("=" * 60)

# Now stitch back together 

In [ ]:
"""
────────────────────────────────────────────────────────────────────────────
Stitches cellpose tile masks → global label image, assigns Xenium transcripts
(from transcripts.zarr.zip) to cells, and produces an AnnData (cells × genes)
with spatial coordinates.

Xenium zarr transcript structure (grids layout):
  root
  ├── grids
  │   └── 0               ← finest resolution, all transcripts
  │       ├── 0,0
  │       ├── 0,1  ...    ← spatial chunks
  │       │   ├── location        float32 (N, 3)  [x_um, y_um, z_um]
  │       │   ├── gene_identity   uint32  (N,)     index into gene_names
  │       │   └── quality_score   uint8   (N,)     Phred Q-score
  └── density
      └── gene
              .attrs['gene_names']   list of gene name strings

Inputs
------
MASK_DIR        : folder produced by the cellpose tiling script
TRANSCRIPTS_ZARR: path to transcripts.zarr.zip
OUTPUT_H5AD     : output AnnData file

Units
-----
Xenium transcript locations are in µm.  morphology_focus is typically
0.2125 µm/pixel.  Adjust PIXEL_SIZE_UM if your experiment differs.
"""

import os
import re
import glob

import numpy as np
import pandas as pd
import tifffile
import zarr
import anndata as ad
from scipy.sparse import csr_matrix

from tqdm.auto import tqdm

# ============================================================
# CONFIG  –– edit these
# ============================================================

MASK_DIR = "cellpose_tiles"

TRANSCRIPTS_ZARR = (
    "../../../Broad_SpatialFoundation/"
    "test_data/10X_Xenium_Ovarian_5k/"
    "transcripts.zarr.zip"
)

OUTPUT_H5AD = "xenium_cellpose.h5ad"

PIXEL_SIZE_UM = 0.2125        # µm per pixel for morphology_focus image

QUALITY_FILTER = True         # drop transcripts with Phred qv < 20
MIN_QV         = 20

MIN_COUNTS_PER_CELL = 3       # drop cells with very few transcripts

# ============================================================
# STEP 1 – infer full image size from tile filenames
# ============================================================

print("Scanning tile directory …")

pattern = re.compile(
    r"mask_y(\d+)_x(\d+)_top(\d+)_left(\d+)\.tif$"
)

tile_files = sorted(glob.glob(os.path.join(MASK_DIR, "mask_y*_x*.tif")))

if not tile_files:
    raise FileNotFoundError(f"No tiles found in {MASK_DIR}")

meta = []
for fp in tqdm(tile_files):
    m = pattern.search(os.path.basename(fp))
    if m:
        y0, x0, top, left = map(int, m.groups())
        tile = tifffile.imread(fp)
        h, w = tile.shape
        meta.append(dict(path=fp, y0=y0, x0=x0,
                         top=top, left=left, h=h, w=w))

df_meta = pd.DataFrame(meta)

df_meta["gy0"] = df_meta["y0"] + df_meta["top"]
df_meta["gx0"] = df_meta["x0"] + df_meta["left"]
df_meta["gy1"] = df_meta["gy0"] + df_meta["h"]
df_meta["gx1"] = df_meta["gx0"] + df_meta["w"]

img_height = int(df_meta["gy1"].max())
img_width  = int(df_meta["gx1"].max())

print(f"  {len(df_meta)} tiles found")
print(f"  Reconstructed canvas: {img_height} × {img_width} px")

print("Stitching masks …")
 
global_mask = np.zeros((img_height, img_width), dtype=np.int64)
 
id_offset = np.int64(0)
 
for _, row in df_meta.iterrows():
    tile = tifffile.imread(row["path"]).astype(np.int64)
 
    # Relabel tile compactly 1..N so offset grows by true cell count only
    unique_labels = np.unique(tile)
    unique_labels = unique_labels[unique_labels > 0]   # drop background
    n_cells_tile  = len(unique_labels)
 
    if n_cells_tile == 0:
        continue
 
    remap = np.zeros(int(tile.max()) + 1, dtype=np.int64)
    remap[unique_labels] = np.arange(1, n_cells_tile + 1, dtype=np.int64) + id_offset
    tile = remap[tile]
 
    gy0, gx0 = int(row["gy0"]), int(row["gx0"])
    gy1 = gy0 + tile.shape[0]
    gx1 = gx0 + tile.shape[1]
 
    region = global_mask[gy0:gy1, gx0:gx1]
    empty  = region == 0
    region[empty] = tile[empty]
    global_mask[gy0:gy1, gx0:gx1] = region
 
    id_offset += n_cells_tile
 
n_cells_total = int(global_mask.max())
print(f"  Total unique cell labels after stitching: {n_cells_total}")
 
# ============================================================
# STEP 2b – seam reconciliation (vectorised)
#
#   Strategy:
#   1. Compute ALL cell centroids in one pass using label-weighted
#      sums (no per-cell array scans).
#   2. For each seam, read the 1-px edge on each side → get the
#      small set of cell IDs that touch that seam.
#   3. Look up their precomputed centroids and find pairs whose
#      centroids are within MERGE_MAX_DIST_PX of each other.
#   4. Merge via union-find, apply with a single LUT pass.
#
#   Cost: O(pixels) for centroids + O(seam_cells²) per seam,
#   where seam_cells is tiny (~tens, not thousands).
# ============================================================
 
print("Reconciling seam cells …")
 
MERGE_MAX_DIST_PX = 150   # cells split at a seam will have fragment
                           # centroids very close (<cell diameter apart)
 
# ── 1. Compute all centroids in one vectorised pass ───────────
print("  Computing global centroids …")
 
max_id = int(global_mask.max())
 
# Use float64 accumulators; index directly into flat arrays
flat = global_mask.ravel()
rows_idx = np.repeat(np.arange(img_height, dtype=np.float64), img_width)
cols_idx = np.tile  (np.arange(img_width,  dtype=np.float64), img_height)
 
cell_sum_r = np.zeros(max_id + 1, dtype=np.float64)
cell_sum_c = np.zeros(max_id + 1, dtype=np.float64)
cell_count = np.zeros(max_id + 1, dtype=np.int64)
 
np.add.at(cell_sum_r, flat, rows_idx)
np.add.at(cell_sum_c, flat, cols_idx)
np.add.at(cell_count, flat, 1)
 
# Avoid divide-by-zero for background (id=0)
nz = cell_count > 0
cell_sum_r[nz] /= cell_count[nz]
cell_sum_c[nz] /= cell_count[nz]
 
# cell_sum_r[i], cell_sum_c[i] = centroid row/col of cell i
print(f"  Centroids computed for {nz.sum() - 1:,} cells")
 
# ── 2. Find seam positions ────────────────────────────────────
seam_rows = sorted(r for r in df_meta["gy0"].unique() if 0 < r < img_height)
seam_cols = sorted(c for c in df_meta["gx0"].unique() if 0 < c < img_width)
 
merge_pairs = []   # (keep_id, drop_id)
 
# ── 3. Match split fragments at each seam ─────────────────────
for sr in tqdm(seam_rows):
    ids_a = set(global_mask[sr - 1, :].ravel().tolist()) - {0}
    ids_b = set(global_mask[sr,     :].ravel().tolist()) - {0}
    if not ids_a or not ids_b:
        continue
    ids_a, ids_b = np.array(list(ids_a)), np.array(list(ids_b))
    # Centroid positions
    ca = np.column_stack([cell_sum_r[ids_a], cell_sum_c[ids_a]])
    cb = np.column_stack([cell_sum_r[ids_b], cell_sum_c[ids_b]])
    # Pairwise distances (small sets, so O(|a|×|b|) is fine)
    dists = np.linalg.norm(ca[:, None] - cb[None, :], axis=-1)
    ia, ib = np.where(dists < MERGE_MAX_DIST_PX)
    for i, j in zip(ia, ib):
        a, b = int(ids_a[i]), int(ids_b[j])
        # keep the one with more pixels
        if cell_count[a] >= cell_count[b]:
            merge_pairs.append((a, b))
        else:
            merge_pairs.append((b, a))
 
for sc in tqdm(seam_cols):
    ids_l = set(global_mask[:, sc - 1].ravel().tolist()) - {0}
    ids_r = set(global_mask[:, sc    ].ravel().tolist()) - {0}
    if not ids_l or not ids_r:
        continue
    ids_l, ids_r = np.array(list(ids_l)), np.array(list(ids_r))
    cl = np.column_stack([cell_sum_r[ids_l], cell_sum_c[ids_l]])
    cr = np.column_stack([cell_sum_r[ids_r], cell_sum_c[ids_r]])
    dists = np.linalg.norm(cl[:, None] - cr[None, :], axis=-1)
    il, ir = np.where(dists < MERGE_MAX_DIST_PX)
    for i, j in zip(il, ir):
        l, r = int(ids_l[i]), int(ids_r[j])
        if cell_count[l] >= cell_count[r]:
            merge_pairs.append((l, r))
        else:
            merge_pairs.append((r, l))
 
print(f"  Seam merge candidates: {len(merge_pairs)}")
 
# ── 4. Union-find + single LUT relabel ───────────────────────
parent = {}
 
def find(x):
    while parent.get(x, x) != x:
        parent[x] = parent.get(parent.get(x, x), x)
        x = parent.get(x, x)
    return x
 
for keep, drop in merge_pairs:
    rk, rd = find(keep), find(drop)
    if rk != rd:
        parent[rd] = rk
 
all_touched = set(parent.keys())
if all_touched:
    lut = np.arange(max_id + 1, dtype=np.int64)
    for old_id in all_touched:
        lut[old_id] = find(old_id)
    global_mask = lut[global_mask]
    n_merged = sum(1 for o in all_touched if lut[o] != o)
    print(f"  Cell fragments merged: {n_merged}")
else:
    print("  No seam merges needed.")
 
n_cells_reconciled = len(np.unique(global_mask)) - 1
print(f"  Cells after reconciliation: {n_cells_reconciled:,}")

In [ ]:
# ============================================================
# STEP 3 – load transcripts from zarr
# ============================================================

print("Loading transcripts from zarr …")

store = zarr.ZipStore(TRANSCRIPTS_ZARR, mode="r")
root  = zarr.open_group(store=store, mode="r")

# Gene name lookup
gene_names = list(root["density"]["gene"].attrs["gene_names"])
print(f"  Gene panel size: {len(gene_names)}")

# Collect all chunks at finest resolution (grids/0)
grid0 = root["grids"]["0"]

all_x   = []
all_y   = []
all_qv  = []
all_gid = []

chunk_keys = sorted(grid0.keys())
print(f"  Reading {len(chunk_keys)} grid chunks …")

for key in tqdm(chunk_keys):
    chunk = grid0[key]
    loc   = chunk["location"][:]          # (N, 3)  x, y, z in µm
    gid   = chunk["gene_identity"][:].ravel()     # (N,)    integer gene index
    qv    = chunk["quality_score"][:].ravel()     # (N,)    Phred score

    all_x.append(loc[:, 0])
    all_y.append(loc[:, 1])
    all_gid.append(gid)
    all_qv.append(qv)

x_um  = np.concatenate(all_x)
y_um  = np.concatenate(all_y)
gid   = np.concatenate(all_gid)
qv    = np.concatenate(all_qv)

print(f"  {len(x_um):,} transcripts loaded")

# Quality filter
if QUALITY_FILTER:
    keep = qv >= MIN_QV
    x_um, y_um, gid, qv = x_um[keep], y_um[keep], gid[keep], qv[keep]
    print(f"  After qv≥{MIN_QV} filter: {len(x_um):,}")

# ============================================================
# STEP 4 – convert µm → pixel and look up cell labels
# ============================================================

print("Assigning transcripts to cells …")

px = np.clip((x_um / PIXEL_SIZE_UM).round().astype(int), 0, img_width  - 1)
py = np.clip((y_um / PIXEL_SIZE_UM).round().astype(int), 0, img_height - 1)

cell_id = global_mask[py, px]

in_cell = cell_id > 0
print(f"  Transcripts in a cell: {in_cell.sum():,} / {len(cell_id):,} "
      f"({100*in_cell.mean():.1f}%)")

# Keep only assigned transcripts
x_um    = x_um[in_cell]
y_um    = y_um[in_cell]
gid     = gid[in_cell]
cell_id = cell_id[in_cell]

# ============================================================
# STEP 5 – build count matrix (cells × genes)
# ============================================================

print("Building count matrix …")

unique_cells  = np.unique(cell_id)
unique_genes  = np.arange(len(gene_names))          # keep full panel

cell_index = {c: i for i, c in enumerate(unique_cells)}

rows = np.array([cell_index[c] for c in cell_id])
cols = gid.astype(int)                              # already 0-based gene indices

counts = csr_matrix(
    (np.ones(len(rows), dtype=np.float32), (rows, cols)),
    shape=(len(unique_cells), len(gene_names)),
)

print(f"  Matrix shape: {counts.shape}  (cells × genes)")

# ============================================================
# STEP 6 – cell spatial centroids (mean transcript position)
# ============================================================

print("Computing cell centroids …")

# Vectorised centroid computation
cell_idx_per_tx = np.array([cell_index[c] for c in cell_id])

cx = np.zeros(len(unique_cells), dtype=np.float64)
cy = np.zeros(len(unique_cells), dtype=np.float64)
cn = np.zeros(len(unique_cells), dtype=np.int64)

np.add.at(cx, cell_idx_per_tx, x_um)
np.add.at(cy, cell_idx_per_tx, y_um)
np.add.at(cn, cell_idx_per_tx, 1)

cx /= cn
cy /= cn

# ============================================================
# STEP 7 – assemble AnnData
# ============================================================

print("Assembling AnnData …")

obs = pd.DataFrame(
    {
        "cell_id"       : unique_cells,
        "x_centroid_um" : cx,
        "y_centroid_um" : cy,
        "x_centroid_px" : cx / PIXEL_SIZE_UM,
        "y_centroid_px" : cy / PIXEL_SIZE_UM,
        "n_transcripts" : np.asarray(counts.sum(axis=1)).ravel(),
    },
    index=[str(c) for c in unique_cells],
)

var = pd.DataFrame(
    {"gene_name": gene_names},
    index=gene_names,
)

adata = ad.AnnData(X=counts, obs=obs, var=var)

# Squidpy / Scanpy spatial convention: obsm["spatial"] = (x, y) in µm
adata.obsm["spatial"] = np.column_stack([cx, cy])

In [ ]:
import re

# Keep only real genes — drop all control/deprecated features
bad_prefixes = (
    "DeprecatedCodeword",
    "NegControlCodeword", 
    "NegControlProbe",
    "UnassignedCodeword",
    "BLANK",
)

is_real_gene = ~adata.var_names.str.startswith(bad_prefixes)
print(f"Removing {(~is_real_gene).sum()} control features, keeping {is_real_gene.sum()} genes")

adata = adata[:, is_real_gene].copy()

In [ ]:
# ── Add control QC metrics ────────────────────────────────────
adata.obs["n_transcripts"]  = np.asarray(adata.X.sum(axis=1)).ravel()  # recompute on real genes only

# ── Reapply cell QC filter ────────────────────────────────────
MIN_COUNTS_PER_CELL = 5

before = adata.n_obs
adata  = adata[adata.obs["n_transcripts"] >= MIN_COUNTS_PER_CELL].copy()
adata = adata[adata.obs.n_transcripts < adata.obs.n_transcripts.quantile(0.995)].copy()
print(f"Cells after min-counts filter: {adata.n_obs}  (dropped {before - adata.n_obs})")

# ── Drop genes with zero counts after filtering ───────────────
sc_before   = adata.n_vars
gene_counts = np.asarray(adata.X.sum(axis=0)).ravel()
adata       = adata[:, gene_counts > 0].copy()
print(f"Genes with ≥1 count: {adata.n_vars}  (dropped {sc_before - adata.n_vars} zeros)")
adata.obsm['spatial_px'] = adata.obs[["x_centroid_px", "y_centroid_px"]].values
# ── Save ──────────────────────────────────────────────────────
adata.write_h5ad("xenium_cellpose.h5ad")
print(f"\nDone — {adata.n_obs:,} cells × {adata.n_vars:,} genes")